## LeNet-5

A CNN model based on LeNet-5 to classify MNIST

In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
# Load MNIST and create train/test split

# Convert images to tensors and normalize with mean and std of MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
]) 

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Number of samples model processes at once during training/testing 
# Model sees 64 images at a time, computes loss and back propagates gradients, then SGD updates weights. 
batch_size = 64 

# Create data loaders to handle batching and shuffling of data during training/testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 60000
Test size: 10000


In [ ]:
model = nn.Sequential(
    # MNIST images are 1x28x28, so the first conv must take 1 input channel. 
    # layer 1 with 32 filters, kernel size 3, (28x28x1 -> 26x26x32) producing 32 feature maps slices of size 28x28
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3),
    nn.ReLU(),


    # layer 2 with 64 filters, kernel size 3, (26x26x32 -> 24x24x64 -> 12x12x64) producing 64 feature maps slices of size 14x14
    nn.Conv2d(32, 64, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Dropout(0.25),

    # dense layer with 128 neurons, input size is 64 feature maps of size 12x12 (after max pooling), output size is 10 for the 10 classes in MNIST
    nn.Flatten(),
    nn.Linear(64 * 12 * 12, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 10),
    # nn.Softmax(dim=1) is covered by loss function.
)


In [4]:
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Total model parameters:", total_params)

Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (3): ReLU()
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Dropout(p=0.25, inplace=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=9216, out_features=128, bias=True)
  (8): ReLU()
  (9): Dropout(p=0.5, inplace=False)
  (10): Linear(in_features=128, out_features=10, bias=True)
)
Total model parameters: 1199882


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer 

In [6]:
# Model training loop with GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

epochs = 30
for epoch in range(epochs):
    model.train() # Set model to training mode (enables dropout, batch norm, etc. if present)
    running_loss = 0.0 
    correct = 0
    total = 0

    # Iterate over mini-batches from the training loader
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images) # Forward pass: compute predicted probabilities for the current batch
        loss = criterion(outputs, labels) # CrossEntropyLoss expects class indices, not one-hot labels
        loss.backward() # Backward pass: compute gradients of the loss with respect to model parameters
        optimizer.step() # Update model parameters based on computed gradients and learning rate

        # Track loss and accuracy for the current epoch
        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1) # Get predicted class 
        correct += (predicted == labels).sum().item() # Count correct predictions in the current batch
        total += labels.size(0)

    train_loss = running_loss / total # Epoch average loss per sample
    train_acc = correct / total # Epoch training accuracy 

    # Evaluate on the test split without computing gradients
    model.eval() # Set model to evaluation mode (disables dropout, batch norm updates, etc.)
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images) # Forward pass on test data to compute predicted probabilities
            predicted = outputs.argmax(dim=1) # Get predicted class
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total


    print(f"Epoch {epoch + 1}/{epochs} - loss: {train_loss:.4f} - train acc: {train_acc:.4f} - test acc: {test_acc:.4f}")

Using device: mps
Epoch 1/30 - loss: 0.1831 - train acc: 0.9450 - test acc: 0.9855
Epoch 2/30 - loss: 0.0792 - train acc: 0.9768 - test acc: 0.9878
Epoch 3/30 - loss: 0.0626 - train acc: 0.9804 - test acc: 0.9898
Epoch 4/30 - loss: 0.0513 - train acc: 0.9840 - test acc: 0.9909
Epoch 5/30 - loss: 0.0422 - train acc: 0.9870 - test acc: 0.9899
Epoch 6/30 - loss: 0.0395 - train acc: 0.9877 - test acc: 0.9922
Epoch 7/30 - loss: 0.0348 - train acc: 0.9883 - test acc: 0.9906
Epoch 8/30 - loss: 0.0289 - train acc: 0.9905 - test acc: 0.9924
Epoch 9/30 - loss: 0.0293 - train acc: 0.9908 - test acc: 0.9930
Epoch 10/30 - loss: 0.0279 - train acc: 0.9907 - test acc: 0.9931
Epoch 11/30 - loss: 0.0233 - train acc: 0.9922 - test acc: 0.9925
Epoch 12/30 - loss: 0.0235 - train acc: 0.9923 - test acc: 0.9924
Epoch 13/30 - loss: 0.0199 - train acc: 0.9933 - test acc: 0.9934
Epoch 14/30 - loss: 0.0187 - train acc: 0.9940 - test acc: 0.9931
Epoch 15/30 - loss: 0.0191 - train acc: 0.9936 - test acc: 0.9927
E